# 04 — Сводная таблица


In [6]:
import os
os.makedirs("predictions", exist_ok=True)

from google.colab import files
uploaded = files.upload()
for fname, content in uploaded.items():
    with open(os.path.join("predictions", fname), "wb") as f:
        f.write(content)
print(f"Загружено файлов: {len(uploaded)}")


Saving gqa_pali_baseline 19.40.54.json to gqa_pali_baseline 19.40.54 (1).json
Saving gqa_pali_finetuned-2.json to gqa_pali_finetuned-2 (1).json
Saving gqa_qwen_full_baseline.json to gqa_qwen_full_baseline (1).json
Saving gqa_qwen_full_finetuned.json to gqa_qwen_full_finetuned (1).json
Saving mmbench_pali_baseline 19.40.54.json to mmbench_pali_baseline 19.40.54 (1).json
Saving mmbench_pali_finetuned-2.json to mmbench_pali_finetuned-2 (1).json
Saving mmbench_qwen_full_baseline.json to mmbench_qwen_full_baseline (1).json
Saving mmbench_qwen_full_finetuned.json to mmbench_qwen_full_finetuned (1).json
Saving gqa_qwen_lora_baseline.json to gqa_qwen_lora_baseline.json
Saving gqa_qwen_lora_finetuned-2.json to gqa_qwen_lora_finetuned-2.json
Saving mmbench_qwen_lora_baseline.json to mmbench_qwen_lora_baseline.json
Saving mmbench_qwen_lora_finetuned-2.json to mmbench_qwen_lora_finetuned-2.json
Загружено файлов: 12


## точность по каждому файлу

In [2]:
import json, re

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower()).strip(".,!?\"'")

def is_soft_match(gold, pred):
    return normalize(gold) in normalize(pred)

def load_acc(filename, soft=False):
    path = os.path.join("predictions", filename)
    if not os.path.exists(path):
        print(f"[пропущено] {filename})
        return None
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)
    if not records:
        return None
    if soft:
        return sum(is_soft_match(r["gold"], r["pred"]) for r in records) / len(records)
    return sum(r["correct"] for r in records) / len(records)


## Сводная таблица

Для GQA-ru показаны две метрики: **строгий ExactMatch** (как считалось во время оценки -
точное совпадение строки) и **мягкое совпадение** (эталонный ответ встречается внутри
ответа модели — ловит случаи вида "Дом находится на правой стороне" при эталоне "справа",
но не ловит другие словоформы вроде "зелёные"/"зеленый")


In [8]:
import pandas as pd

rows = [
    {"run": "Qwen2-VL-2B — baseline", "method": "—",
     "GQA-ru (EM)": load_acc("gqa_qwen_lora_baseline.json"), "GQA-ru (мягкий)": load_acc("gqa_qwen_lora_baseline.json", soft=True),
     "MMBench-ru": load_acc("mmbench_qwen_lora_baseline.json")},
    {"run": "Qwen2-VL-2B — LoRA", "method": "LoRA",
     "GQA-ru (EM)": load_acc("gqa_qwen_lora_finetuned-2.json"), "GQA-ru (мягкий)": load_acc("gqa_qwen_lora_finetuned-2.json", soft=True),
     "MMBench-ru": load_acc("mmbench_qwen_lora_finetuned-2.json")},
    {"run": "Qwen2-VL-2B — baseline (bf16, для сравнения)", "method": "—",
     "GQA-ru (EM)": load_acc("gqa_qwen_full_baseline.json"), "GQA-ru (мягкий)": load_acc("gqa_qwen_full_baseline.json", soft=True),
     "MMBench-ru": load_acc("mmbench_qwen_full_baseline.json")},
    {"run": "Qwen2-VL-2B — полный файнтюнинг декодера", "method": "full decoder FT",
     "GQA-ru (EM)": load_acc("gqa_qwen_full_finetuned.json"), "GQA-ru (мягкий)": load_acc("gqa_qwen_full_finetuned.json", soft=True),
     "MMBench-ru": load_acc("mmbench_qwen_full_finetuned.json")},
    {"run": "PaliGemma2-3B — baseline", "method": "—",
     "GQA-ru (EM)": load_acc("gqa_pali_baseline 19.40.54.json"), "GQA-ru (мягкий)": load_acc("gqa_pali_baseline 19.40.54.json", soft=True),
     "MMBench-ru": load_acc("mmbench_pali_baseline 19.40.54.json")},
    {"run": "PaliGemma2-3B — LoRA", "method": "LoRA",
     "GQA-ru (EM)": load_acc("gqa_pali_finetuned-2.json"), "GQA-ru (мягкий)": load_acc("gqa_pali_finetuned-2.json", soft=True),
     "MMBench-ru": load_acc("mmbench_pali_finetuned-2.json")},
]
summary = pd.DataFrame(rows)
summary


,run,method,GQA-ru (EM),GQA-ru (мягкий),MMBench-ru
0,Qwen2-VL-2B — baseline,—,0.300,0.310,0.690
1,Qwen2-VL-2B — LoRA,LoRA,0.190,0.310,0.680
2,"Qwen2-VL-2B — baseline (bf16, для сравнения)",—,0.270,0.275,0.695
3,Qwen2-VL-2B — полный файнтюнинг декодера,full decoder FT,0.230,0.245,0.680
4,PaliGemma2-3B — baseline,—,0.140,0.165,0.535
5,PaliGemma2-3B — LoRA,LoRA,0.085,0.150,0.235
